<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "044"
iso_country = "Bahamas"
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2024
gdp = 15.830*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Bahamas': 

Number of trade records present in dataset: 17268
Number of partner countries present in dataset: 101
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$4031.1m
Total weight within the dataset: 3925.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,044,Bahamas,124,Canada,Import,Sea,460290,"460290 | Basketwork, wickerwo",4602,"4602 | Basketwork, wickerwo",46,46 | Manufactures of stra,31923,"Manufactures of straw, of esparto or of other ...",Wood and Paper,30,LUM - Wood products,544.3251,NaN,544.3251,kg,1.262055,1.262055,88.126234,3044.0,1891.453324,166.686658,['all'],['all'],0.486381,0.485196,0.485315,0.485315,0.485911,0.485951,0.508019,0.885227,0.546659,0.511588,0.513563,0.529894,0.497917,0.864066,0.580340,0.533786,0.522446,0.545748,0.488804,0.956158,0.691820,0.629935,0.586060,0.567658,0.479894,1.021810,0.822039,0.768829,0.625468,0.606884,0.471088,1.060101,1.009136,1.020677,0.662096,0.636247,0.462124,1.008870,0.990425,1.057507,0.838590,0.647082,0.453297,0.900914,0.931955,1.046636,0.810095,0.653962,0.444731,0.809967,0.830577,0.965419,0.798901,0.659070,31923
1,1,044,Bahamas,124,Canada,Import,Sea,480261,480261 | Uncoated paper and p,4802,4802 | Uncoated paper and p,48,48 | Paper and paperboard,32129,"Other uncoated paper and paperboard, of a kind...",Wood and Paper,31,"PPP - Paper products, publishing",1044.7750,NaN,1044.7750,NaN,0.000000,NaN,1056.058570,3044.0,1891.453324,1997.485493,"['Container', 'General cargo']","['Container', 'General cargo']",5.565691,5.548350,5.550032,5.550032,5.559246,5.559246,5.857791,10.895269,6.388847,5.942148,5.965085,6.139529,5.721769,10.602723,6.756163,6.187254,6.024254,6.345516,5.602249,11.758669,8.227847,7.362398,6.781248,6.633704,5.484937,12.301249,9.856704,9.130830,7.394831,7.149070,5.369997,12.920007,12.483292,12.382520,7.802369,7.528487,5.252837,12.560068,12.230811,13.174892,10.335333,7.669247,5.133259,10.872881,11.488232,12.883368,9.742554,7.759445,5.021114,9.696054,9.955762,11.844553,9.450330,7.820518,32129


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $867.1m (21.5%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
22 | Beverages, spirits a  140392434.8  773960718.1      3.5     19.7
02 | Meat and edible meat  117042339.4   45652950.4      2.9      1.2
21 | Miscellaneous edible   86256546.5   25558885.3      2.1      0.7
19 | Preparations of cere   83362825.9   31744574.2      2.1      0.8
04 | Dairy produce; birds   75592356.2   37461349.2      1.9      1.0
07 | Vegetables and certa   75396262.2   90117851.9      1.9      2.3
08 | Fruit and nuts, edib   50684757.8   46006243.3      1.3      1.2
20 | Preparations of vege   48448327.4   33912300.0      1.2      0.9
16 | Meat, fish, crustace   46737848.1   10451647.2      1.2      0.3
03 | Fish and crustaceans   29861544.2    5584502.3      0.7      0.1
17 | Sugars and sugar con   21241147.5   32217971.5      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $773.8m (19.2%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  414548740.2  875540664.4     10.3     22.3
Agriculture       353170582.0  268141935.0      8.8      6.8
Fishing             6044863.2    1375249.9      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $662.2m (16.4%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     292949323.1  153176107.2      7.3      3.9
OMT - Meat products n.e.c.     121513931.0   47746929.8      3.0      1.2
V_F - Vegetables, fruit, nuts   89560121.2  101660979.9      2.2      2.6
MIL - Dairy products            58385207.1   24534704.1      1.4      0.6
CMT - Bovine meat prods         32250553.8    6162478.1      0.8      0.2
OCR - Crops n.e.c.              24244524.3   11656373.3      0.6      0.3
VOL - Vegetable oils and fats   18148220.8   10794102.7      0.5      0.3
PCR - Processed rice            11361936.1   19735683.1      0.3      0.5
SGR - Sugar                      8786152.7   19102277.9      0.2      0.5
PDR - Paddy rice                 1864283.2    3238259.9      0.0      0.1
OAP - Animal products n.e.c.     1287166.6    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 21.5% 
GTAP 16.4% 
EORA 19.2%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_23063/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,044,Bahamas,124,Canada,Import,Sea,460290,"460290 | Basketwork, wickerwo",4602,"4602 | Basketwork, wickerwo",46,46 | Manufactures of stra,31923,"Manufactures of straw, of esparto or of other ...",Wood and Paper,30,LUM - Wood products,544.3251,NaN,544.3251,kg,1.262055,1.262055,88.126234,3044.0,1891.453324,166.686658,['all'],['all'],0.486381,0.485196,0.485315,0.485315,0.485911,0.485951,0.508019,0.885227,0.546659,0.511588,0.513563,0.529894,0.497917,0.864066,0.580340,0.533786,0.522446,0.545748,0.488804,0.956158,0.691820,0.629935,0.586060,0.567658,0.479894,1.021810,0.822039,0.768829,0.625468,0.606884,0.471088,1.060101,1.009136,1.020677,0.662096,0.636247,0.462124,1.008870,0.990425,1.057507,0.838590,0.647082,0.453297,0.900914,0.931955,1.046636,0.810095,0.653962,0.444731,0.809967,0.830577,0.965419,0.798901,0.659070,31923,NaN
1,1,044,Bahamas,124,Canada,Import,Sea,480261,480261 | Uncoated paper and p,4802,4802 | Uncoated paper and p,48,48 | Paper and paperboard,32129,"Other uncoated paper and paperboard, of a kind...",Wood and Paper,31,"PPP - Paper products, publishing",1044.7750,NaN,1044.7750,NaN,0.000000,NaN,1056.058570,3044.0,1891.453324,1997.485493,"['Container', 'General cargo']","['Container', 'General cargo']",5.565691,5.548350,5.550032,5.550032,5.559246,5.559246,5.857791,10.895269,6.388847,5.942148,5.965085,6.139529,5.721769,10.602723,6.756163,6.187254,6.024254,6.345516,5.602249,11.758669,8.227847,7.362398,6.781248,6.633704,5.484937,12.301249,9.856704,9.130830,7.394831,7.149070,5.369997,12.920007,12.483292,12.382520,7.802369,7.528487,5.252837,12.560068,12.230811,13.174892,10.335333,7.669247,5.133259,10.872881,11.488232,12.883368,9.742554,7.759445,5.021114,9.696054,9.955762,11.844553,9.450330,7.820518,32129,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $525.0m of $4031.1m (13.0%)
Volume: 371.8 of 3925.6 million kilograms (9.5%)
Value-to-Weight: 1.41 from 1.03 US$ per kg (+37.5%)
Average distance to market (volume-weighted): 1366.3 from 1276.0 nm (+7.1%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    372  190227998.7  104699022.1      4.7      2.7
1           Vegetables    295   95831964.8  107019679.6      2.4      2.7
2        Meat and Prep    172   90612476.3   39831597.6      2.2      1.0
3                Dairy    107   57348716.5   27855578.1      1.4      0.7
4               Fruits    180   48784573.3   45147871.2      1.2      1.2
5      Sugar and Honey     86   21311054.9   32238014.0      0.5      0.8
6        Fats and Oils     88   19893009.0   11272293.2      0.5      0.3
7  Unprocessed Cereals     33     994402.3    3759621.0      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,10608924.4,19888373.5,11591577.4,10764961.1,10810676.8,11128447.9,9708337.9,23644755.7,22862231.1,22618904.6,14189194.7,13695777.7,9063592.4,17629567.2,18116305.0,21573052.3,17224218.9,14235998.8
1,Fats and Oils,3696902.8,6693647.2,4014100.9,3736892.6,3751393.8,3866744.7,3406194.1,7851225.2,7623792.7,7617796.4,4882144.5,4695763.8,3198959.5,6006267.8,6149986.1,7273134.5,5871429.9,4870335.1
2,Fruits,13098870.6,24669595.9,14326179.8,13297903.6,13354642.0,13744948.9,11976885.2,29284572.6,28359063.2,28021771.2,17549364.7,16939984.5,11173707.8,21823418.1,22426478.3,26754437.4,21296073.8,17612258.4
3,Meat and Prep,21405031.8,40423514.7,23424420.5,21736694.6,21829763.6,22465719.2,19561395.3,47971991.7,46482397.5,45893173.8,28706293.9,27713957.7,18241495.9,35718324.2,36708179.2,43831444.8,34842193.5,28817570.6
4,Processed Cereals,43148520.1,77991127.6,46751339.1,43601222.2,43776995.4,45119750.2,39760178.4,92849742.4,89530120.2,89401273.3,57414818.4,54759220.2,37342104.7,70491533.0,72336801.0,85226596.7,68906798.7,56773754.4
5,Sugar and Honey,13498723.5,25135971.0,14724844.4,13685806.1,13743235.6,14150550.4,12369629.6,29848966.6,28885112.6,28621116.2,18057382.8,17363772.1,11562124.3,22370084.3,22977673.4,27322244.5,21834059.9,18038026.5
6,Unprocessed Cereals,1074977.3,1899091.7,1158433.7,1083359.8,1087578.6,1121829.5,994776.1,2262592.2,2181739.5,2189838.2,1431032.5,1349023.4,937701.3,1740953.5,1784839.9,2089492.6,1699893.3,1395856.4
7,Vegetables,37437731.8,69525911.8,40822164.2,37952042.5,38110606.3,39240362.3,34325137.3,82613717.1,79772359.4,79152324.1,49922607.0,48129516.3,32092707.9,61888530.5,63575564.7,75514978.6,60481170.5,50003039.4


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.16,0.02,0.00,0.00,0.01,0.24,0.23,0.23,0.08,0.07,0.15,0.16,0.22,0.14,0.09
1,Fats and Oils,0.15,0.02,0.00,0.00,0.01,0.22,0.21,0.21,0.07,0.06,0.14,0.15,0.20,0.13,0.08
2,Fruits,0.24,0.03,0.00,0.01,0.01,0.35,0.34,0.33,0.11,0.10,0.22,0.23,0.32,0.21,0.13
3,Meat and Prep,0.21,0.02,0.00,0.00,0.01,0.31,0.30,0.29,0.10,0.09,0.19,0.20,0.28,0.18,0.12
4,Processed Cereals,0.18,0.02,0.00,0.00,0.01,0.28,0.26,0.26,0.09,0.08,0.17,0.18,0.25,0.17,0.10
5,Sugar and Honey,0.55,0.06,0.01,0.01,0.03,0.82,0.77,0.76,0.27,0.23,0.51,0.54,0.74,0.48,0.30
6,Unprocessed Cereals,0.83,0.08,0.01,0.01,0.05,1.27,1.19,1.20,0.44,0.36,0.81,0.85,1.16,0.77,0.46
7,Vegetables,0.33,0.04,0.01,0.01,0.02,0.50,0.47,0.47,0.16,0.14,0.31,0.33,0.45,0.30,0.19
